# Automation Bot for Queue Booking System
Run the cells sequentially to execute the bot step-by-step.

In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from selenium import webdriver
from selenium.webdriver.chrome.options import Options

import time
import pandas as pd
import bs4
import os

print("✅ Libraries imported successfully.")

✅ Libraries imported successfully.


In [26]:
# ==========================================
# Initial Variables (ตั้งค่าเริ่มต้น)
# ==========================================
TARGET_URL = "http://localhost:3000/"  # ใส่ URL ที่ได้จากการ Deploy

TARGET_DUTY = "ลบกระดานปิดหน้าต่าง"           # หน้าที่ที่ต้องการจอง
TARGET_NAME = "นายทดสอบ ระบบอัตโนมัติ"  # ชื่อผู้จอง
DELAY_SECONDS = 1                # เวลาหน่วง (วินาที)

print("✅ Variables initialized.")

✅ Variables initialized.


In [27]:
options = webdriver.ChromeOptions()
from bot_script import DELAY_SECONDS
# เว็บไม่ปิดอัตโนมัติ
options.add_experimental_option("detach", True)
options.add_experimental_option("excludeSwitches", ['enable-automation'])
try:
    # ลองใช้ทรัพยากรณ์ chrome, chromedriverว่าเวอร์ชันตรงไหม
    driver = webdriver.Chrome(options=options)
except Exception as e:
    print("อาจจะเวอร์ชั่น Chromedriver ไม่ตรง")
    print("Fix version chrome driver")
    # ถ้าเวอร์ชันไม่ตรงก็ install chrome driverใหม่
    from selenium.webdriver.chrome.service import Service
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

print("ตั้งค่าChromeเสร็จ")

driver.get(TARGET_URL) # เปิดเว็บด้วยChrome

time.sleep(DELAY_SECONDS)
# ซูมออกจากหน้าเว็บให้ข้อมูลโหลดทั้งหน้า
driver.execute_script("document.body.style.zoom='70%'")
print("✅ Browser launched.")

ตั้งค่าChromeเสร็จ
✅ Browser launched.


In [23]:
# Switch frame logic (for Apps Script Web Apps)
try:
     WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "sandboxFrame")))
     print("Switched to sandboxFrame")
     WebDriverWait(driver, 10).until(EC.frame_to_be_available_and_switch_to_it((By.ID, "userHtmlFrame")))
     print("Switched to userHtmlFrame")
except:
     print("Could not switch frames, or frames not present. Trying to interact directly.")

Could not switch frames, or frames not present. Trying to interact directly.


In [29]:
TARGET_DATE = "2026-01-21"          # วันที่ที่ต้องการจอง (YYYY-MM-DD)

# 2. เลือกวันที่
print(f"📅 ระบุวันที่: {TARGET_DATE}")
date_input = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.XPATH, '//*[@id="date"]'))
)

# ใช้ JavaScript ในการป้อนวันที่เพื่อให้ชัวร์ (เนื่องจาก input type="date" ในแต่ละท้องถิ่นอาจรับ format ต่างกัน)
driver.execute_script(f"arguments[0].value = '{TARGET_DATE}';", date_input)
# กระตุ้นให้ JavaScript ของหน้าเว็บรับรู้การเปลี่ยนแปลง
driver.execute_script("arguments[0].dispatchEvent(new Event('change'))", date_input)

# รอให้ระบบดึงข้อมูลหน้าที่ (Dropdown จะถูก enable)
time.sleep(DELAY_SECONDS) 

📅 ระบุวันที่: 2026-01-21


In [30]:
# 3. Select Duty
print(f"🧹 Selecting Duty: {TARGET_DUTY}")
duty_select_elem = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.XPATH, '//*[@id="duty"]'))
)

# Wait until it's enabled
WebDriverWait(driver, 10).until(
    lambda d: duty_select_elem.get_attribute("disabled") is None
)

select = Select(duty_select_elem)
try:
    select.select_by_visible_text(TARGET_DUTY)
    print(f"✅ Selected: {TARGET_DUTY}")
except:
    print(f"❌ Duty '{TARGET_DUTY}' not found or already booked!")
    options = [o.text for o in select.options]
    print(f"Available options: {options}")

time.sleep(DELAY_SECONDS)

🧹 Selecting Duty: ลบกระดานปิดหน้าต่าง
✅ Selected: ลบกระดานปิดหน้าต่าง


In [31]:
# 4. Input Name
print(f"✍️ Entering Name: {TARGET_NAME}")
name_input = driver.find_element(By.XPATH, '//*[@id="name"]')
name_input.clear()
name_input.send_keys(TARGET_NAME)
time.sleep(DELAY_SECONDS)

✍️ Entering Name: นายทดสอบ ระบบอัตโนมัติ


In [20]:
# 5. Submit
print("✅ Clicking Submit...")
submit_btn = driver.find_element(By.XPATH, '//*[@id="submitBtn"]')
submit_btn.click()

# 6. Wait for result
try:
    success_msg = WebDriverWait(driver, 10).until(
        EC.visibility_of_element_located((By.CLASS_NAME, "success"))
    )
    print(f"🎉 Success: {success_msg.text}")
except:
    print("⚠️ Success message not detected or timed out.")

✅ Clicking Submit...
🎉 Success: บันทึกการจองเรียบร้อย (Local Storage)


In [ ]:
# Close Browser (Optional)
# driver.quit()